In [ ]:
# Cell 0
!pip install -q "transformers==4.46.3" "trl==0.12.2" "peft==0.13.2" "accelerate==1.1.1" "datasets==3.1.0"
!pip install -q -U bitsandbytes

import torch, transformers, peft, trl, bitsandbytes as bnb, accelerate
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"bitsandbytes : {bnb.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 1 — locate the dataset

# Kaggle mounts attached datasets read-only under /kaggle/input/<slug>/.
# Slug for "FanfictionFT" is typically "fanfictionft" (lowercased).
# Run this once to confirm where your three files actually are.

import os
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

print(" ok")

In [ ]:
# Cell 2 — config

import json, math
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ── Model ───────────────────────────────────────────────────────────────────
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

# ── Paths ───────────────────────────────────────────────────────────────────
# IMPORTANT: adjust DATA_DIR if Cell 1 showed a different folder name.
DATA_DIR   = Path("//kaggle/input/datasets/ematoda/fanfictionft")
OUTPUT_DIR = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE         = DATA_DIR / "train_merged.jsonl"
EVAL_FILE          = DATA_DIR / "eval_merged.jsonl"
SYSTEM_PROMPT_PATH = DATA_DIR / "system_default.txt"

assert TRAIN_FILE.exists(),         f"Not found: {TRAIN_FILE}"
assert EVAL_FILE.exists(),          f"Not found: {EVAL_FILE}"
assert SYSTEM_PROMPT_PATH.exists(), f"Not found: {SYSTEM_PROMPT_PATH}"

SYSTEM_PROMPT = SYSTEM_PROMPT_PATH.read_text(encoding="utf-8").strip()
print(f"System prompt length: {len(SYSTEM_PROMPT)} chars")
print(f"--- Preview ---\n{SYSTEM_PROMPT[:200]}")

# ── HF token (for gated Mistral model) ──────────────────────────────────────
# Add your HF token under Add-ons → Secrets, label it HF_TOKEN.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF token loaded from Kaggle Secrets.")
except Exception as e:
    print(f"No HF_TOKEN secret found ({e}). Continuing without — may fail "
          f"if the model is gated and not cached.")

# ── QLoRA ───────────────────────────────────────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ── Training (T4/P100 16GB-friendly) ────────────────────────────────────────
MAX_SEQ_LENGTH   = 512
PER_DEVICE_BATCH = 2
GRAD_ACCUM       = 8
NUM_EPOCHS       = 3
LEARNING_RATE    = 2e-4
LR_SCHEDULER     = "cosine"
WARMUP_RATIO     = 0.05
MAX_GRAD_NORM    = 0.3
WEIGHT_DECAY     = 0.001
SAVE_STEPS       = 100
EVAL_STEPS       = 100
LOGGING_STEPS    = 10

# ── W&B disabled ───────────────────────────────────────────────────────────
os.environ["WANDB_MODE"] = "disabled"

print(f"\nEffective batch size : {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"Max sequence length  : {MAX_SEQ_LENGTH} tokens")

In [ ]:
# Cell 3 — load & format data

def load_jsonl(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def format_row(row: dict) -> dict:
    text = (
        f"<s>[INST] {SYSTEM_PROMPT}\n\n{row['instruction']} [/INST] "
        f"{row['output']}</s>"
    )
    return {"text": text}

train_raw = load_jsonl(TRAIN_FILE)
eval_raw  = load_jsonl(EVAL_FILE)

train_dataset = Dataset.from_list([format_row(r) for r in train_raw])
eval_dataset  = Dataset.from_list([format_row(r) for r in eval_raw])

print(f"Train examples : {len(train_dataset)}")
print(f"Eval examples  : {len(eval_dataset)}")
print(f"\nSample (first 600 chars):\n{train_dataset[0]['text'][:600]}")

In [ ]:
# Cell 4 — load model in 4-bit

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print(f"Model loaded. Total params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"VRAM used after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 5 — attach LoRA

lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = LORA_TARGETS,
    bias           = "none",
    task_type      = "CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}  ({100 * trainable / total:.2f}%)")

In [ ]:
# Cell 6 — TrainingArguments

# Decide bf16 vs fp16 by GPU. P100 (sm_60) has no bf16; T4 (sm_75) has very
# limited bf16. So on Kaggle's free GPUs → fp16. Newer GPUs → bf16.
USE_BF16 = torch.cuda.is_bf16_supported()
USE_FP16 = not USE_BF16
print(f"Using bf16: {USE_BF16} | fp16: {USE_FP16}")

training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    per_device_train_batch_size = PER_DEVICE_BATCH,
    per_device_eval_batch_size  = PER_DEVICE_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs            = NUM_EPOCHS,
    gradient_checkpointing      = True,        # ON — saves VRAM
    fp16                        = USE_FP16,
    bf16                        = USE_BF16,
    optim                       = "paged_adamw_8bit",
    dataloader_pin_memory       = True,
    dataloader_num_workers      = 2,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_ratio                = WARMUP_RATIO,
    max_grad_norm               = MAX_GRAD_NORM,
    weight_decay                = WEIGHT_DECAY,
    evaluation_strategy         = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    logging_steps               = LOGGING_STEPS,
    report_to                   = "none",
    group_by_length             = False,
    torch_compile               = False,
)

steps_per_epoch = math.ceil(len(train_dataset) / (PER_DEVICE_BATCH * GRAD_ACCUM))
total_steps     = steps_per_epoch * NUM_EPOCHS
print(f"Steps per epoch : {steps_per_epoch}")
print(f"Total steps     : {total_steps}")

In [ ]:
# Cell 7 — quick GPU sanity check

import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,utilization.gpu,memory.used,memory.total",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(result.stdout.strip())
print(f"BF16 supported : {torch.cuda.is_bf16_supported()}")
print(f"CUDA version   : {torch.version.cuda}")
print(f"PyTorch        : {torch.__version__}")

In [ ]:
# Cell 8 — train

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = eval_dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    packing            = False,
    args               = training_args,
)

print(f"VRAM before training : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Starting training...\n")

trainer.train()
print("\nTraining complete.")

In [ ]:
# Cell 9 — save adapter & quick inference test

import shutil  # at top of cell — was a bug in the original

ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

adapter_size_mb = sum(f.stat().st_size for f in ADAPTER_DIR.iterdir()) / 1e6
print(f"Adapter saved to : {ADAPTER_DIR}")
print(f"Adapter size     : {adapter_size_mb:.1f} MB")

# Zip for easy download from the notebook's Output tab
shutil.make_archive("/kaggle/working/final_adapter", "zip", str(ADAPTER_DIR))
print("Adapter zipped: /kaggle/working/final_adapter.zip")

# --- Quick inference test ---------------------------------------------------
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = {"": "cuda:0"},
    trust_remote_code   = True,
    torch_dtype         = torch.float16,
)
inf_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
inf_model.eval()

def generate_story(prompt: str, max_new_tokens: int = 1200) -> str:
    formatted = f"<s>[INST] {SYSTEM_PROMPT}\n\n{prompt} [/INST]"
    inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.8,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

test_prompt = ("Write a short story about an astronaut who discovers the Mars "
               "colony has been keeping a secret from Earth.")
print(generate_story(test_prompt))

In [1]:
   !nvidia-smi

Sun May 17 13:24:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----